In [ ]:
# Copyright (c) 2025 Microsoft Corporation.
import os
from pathlib import Path

import pandas as pd
from pydantic import SecretStr
from rich import print as rich_print

from benchmark_qed.autoe.reference_scores import (
    get_reference_scores,
    summarize_reference_scores,
)
from benchmark_qed.cli.utils import print_df
from benchmark_qed.config.llm_config import LLMConfig, LLMProvider
from benchmark_qed.config.model.score import Criteria, reference_scores_criteria
from benchmark_qed.llm.factory import ModelFactory

# Experiment 11e: AutoE Reference-based Scoring

Reference-based scoring evaluates RAG-generated answers by comparing them against a **reference answer** (a gold-standard or best-available response) using criteria such as correctness and completeness. This approach is especially useful when a strong reference exists and you want to measure how closely a generated answer matches it.

This notebook demonstrates how to:
1. Configure an LLM judge for reference-based evaluation
2. Score multiple RAG methods against a reference across multiple question sets
3. Aggregate and compare per-method and per-criteria mean scores
4. Visualize the results to compare RAG methods at a glance

**Comparison with other AutoE evaluation modes:**

| Mode | When to use | Output |
|------|------------|--------|
| **Pairwise (win rate)** | No reference answer available; relative comparison | Win/lose/tie rates |
| **Reference-based** *(this notebook)* | Reference answer available; absolute quality measurement | Numeric scores (1–10) |
| **Assertion-based** | Verifiable factual claims available; fact-checking focus | Pass/fail per assertion |

Reference-based scoring uses the same counterbalanced trial design as pairwise evaluation to mitigate LLM position bias: in half the trials the reference answer is presented first, and in the other half the generated answer is presented first.

In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
%load_ext dotenv
%dotenv

## Configuration

In [ ]:
# Config LLM model to be used as judge
llm_config = LLMConfig(
    model="gpt-4.1",
    api_key=SecretStr(os.environ["OPENAI_API_KEY"]),
    llm_provider=LLMProvider.OpenAIChat,
    concurrent_requests=32,
    call_args={"temperature": 0.0, "seed": 42},
)
llm_client = ModelFactory.create_chat_model(llm_config)

In [ ]:
# Config conditions for evaluation
reference = "lazygraphrag"  # RAG method to use as the reference (gold-standard)
generated_rags = [
    "vector_rag",
    "graphrag_global",
]  # RAG methods to evaluate against the reference
question_sets = ["activity_global", "activity_local"]
trials = 4  # number of trials per (question, criteria) pair. Must be an even number to support counterbalancing.
score_min = 1
score_max = 10

input_dir = "./example_answers"
output_dir = Path("./output/reference_scores")
output_dir.mkdir(parents=True, exist_ok=True)

# Load default criteria (correctness and completeness).
# You can also define your own criteria as a list of Criteria objects, for example:
# criteria = [
#     Criteria(name="relevance", description="How relevant is the answer to the question?"),
#     Criteria(name="depth",     description="How deeply does the answer address the question?"),
# ]
criteria = reference_scores_criteria()
rich_print(f"Evaluating {len(generated_rags)} RAG method(s) against reference: [bold]{reference}[/bold]")
rich_print(f"Question sets : {question_sets}")
rich_print(f"Criteria      : {[c.name for c in criteria]}")
rich_print(f"Trials        : {trials} (counterbalanced)")

## Reference-based Evaluation

For each combination of RAG method and question set, we:
1. Load the reference answers and the generated answers
2. Use the LLM judge to score each generated answer against the reference on every criterion
3. Aggregate results into per-question and per-method summaries

The LLM judge scores each answer on a scale from `score_min` to `score_max` (default 1–10). Trials are counterbalanced so the reference is presented as "Answer 1" in half the trials and "Answer 2" in the other half, reducing position bias.

In [ ]:
all_results = []
all_summaries = []

for question_set in question_sets:
    rich_print(f"Processing question set: [bold]{question_set}[/bold]")

    reference_answers = pd.read_json(f"{input_dir}/{reference}/{question_set}.json")
    rich_print(f"  Loaded {len(reference_answers)} reference answers from {reference}")

    for generated in generated_rags:
        rich_print(f"  Scoring [bold]{generated}[/bold] vs. reference ({reference})...")

        generated_answers = pd.read_json(f"{input_dir}/{generated}/{question_set}.json")

        result = get_reference_scores(
            llm_client=llm_client,
            llm_config=llm_config,
            reference_answers=reference_answers,
            generated_answers=generated_answers,
            criteria=criteria,
            trials=trials,
            score_min=score_min,
            score_max=score_max,
            include_score_id_in_prompt=True,
            question_id_key="question_id",
        )

        # Save raw scores for this method and question set
        qs_output_dir = output_dir / question_set
        qs_output_dir.mkdir(parents=True, exist_ok=True)
        result.to_csv(
            qs_output_dir / f"{generated}_reference_scores.csv", index=False
        )

        # Summarize scores (mean and std per criterion)
        summary_df = summarize_reference_scores(result)
        summary_df["question_set"] = question_set
        summary_df["reference"] = reference
        summary_df["generated"] = generated
        summary_df.to_csv(
            qs_output_dir / f"{generated}_reference_summary.csv", index=False
        )

        # Report per-criteria summary
        for _, row in summary_df.iterrows():
            rich_print(
                f"    {row['criteria']}: mean={row['mean']:.2f}, std={row['std']:.2f}"
            )

        all_results.append(result)
        all_summaries.append(summary_df)

rich_print("[bold green]Reference-based evaluation complete.[/bold green]")

## Results Summary

The table below shows the mean reference score (and standard deviation) for each RAG method, question set, and criterion. Scores range from `score_min` to `score_max` (default 1–10); higher scores indicate closer alignment with the reference answers on that criterion.

In [ ]:
# Combine all summaries into a single DataFrame
all_summary_df = pd.concat(all_summaries, ignore_index=True)
all_summary_df = all_summary_df.sort_values(
    ["question_set", "criteria", "mean"], ascending=[True, True, False]
).reset_index(drop=True)

# Save combined summary
all_summary_df.to_csv(output_dir / "reference_scores_summary.csv", index=False)

print_df(
    all_summary_df[["question_set", "generated", "criteria", "mean", "std"]],
    f"Reference Scores Summary (reference: {reference})",
)

# Pivot table for easy cross-method comparison
pivot_summary = all_summary_df.pivot_table(
    index=["question_set", "generated"],
    columns="criteria",
    values="mean",
).reset_index()
pivot_summary.to_csv(output_dir / "reference_scores_pivot_summary.csv", index=False)

print_df(pivot_summary, "Mean Reference Scores — Pivot View (by question set and method)")

## Visualization

The chart below compares mean reference scores across all evaluated RAG methods, question sets, and criteria.

In [ ]:
import matplotlib.pyplot as plt

from benchmark_qed.autoe.visualization.utils import (
    add_value_labels,
    calculate_bar_width,
    format_method_name,
    format_question_set_name,
    get_color_palette,
    save_figure,
    setup_grid,
    setup_plot_style,
)

setup_plot_style()

criteria_names = all_summary_df["criteria"].unique()
methods = sorted(all_summary_df["generated"].unique())
question_set_list = sorted(all_summary_df["question_set"].unique())

fig, axes = plt.subplots(
    1, len(question_set_list), figsize=(7 * len(question_set_list), 5), sharey=True
)
if len(question_set_list) == 1:
    axes = [axes]

colors = get_color_palette(len(criteria_names))
width = calculate_bar_width(len(criteria_names))
x = range(len(methods))

for ax, question_set in zip(axes, question_set_list):
    qs_data = all_summary_df[all_summary_df["question_set"] == question_set]
    for i, criterion in enumerate(criteria_names):
        values = [
            qs_data.loc[
                (qs_data["generated"] == m) & (qs_data["criteria"] == criterion),
                "mean",
            ].values[0]
            if len(
                qs_data.loc[
                    (qs_data["generated"] == m) & (qs_data["criteria"] == criterion)
                ]
            ) > 0
            else 0
            for m in methods
        ]
        bars = ax.bar(
            [pos + width * i for pos in x],
            values,
            width,
            label=criterion.title(),
            color=colors[i],
            alpha=0.8,
        )
        add_value_labels(ax, bars, format_str="{:.1f}")

    ax.set_title(format_question_set_name(question_set))
    ax.set_xlabel("RAG Method")
    ax.set_ylabel(f"Mean Score ({score_min}–{score_max})")
    ax.set_xticks([pos + width * (len(criteria_names) - 1) / 2 for pos in x])
    ax.set_xticklabels(
        [format_method_name(m) for m in methods], rotation=45, ha="right"
    )
    ax.set_ylim(0, score_max + 1)
    ax.legend(loc="upper left")
    setup_grid(ax)

fig.suptitle(
    f"Reference-based Scores by RAG Method (reference: {format_method_name(reference)})",
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()
save_figure(fig, output_dir / "reference_scores_chart.png")
plt.show()

rich_print("Model usage statistics:")
rich_print(llm_client.get_usage())